# Timed-Release Cooperative Roles

Train and inspect the 8-ant shared-writes cooperative checkpoint with fixed release-rank timing.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


/home/juan/.local/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


{'jax_already_imported': False,
 'jax_preallocate': 'false',
 'jax_memory_fraction': '0.35',
 'jax_allocator': 'platform',
 'memory_trimmed': True,
 'disk_free_gb': 822.12,
 'disk_used_percent': 5.1,
 'current_pid': 12822,
 'safe_cleanup_candidate_count': 156,
 'safe_cleanup_candidate_gb': 0.018,
 'top_memory_processes': [{'pid': 5077,
   'ppid': 4684,
   'rss_mb': 608.5,
   'command': '/snap/code/247/usr/share/code/code /home/juan/.vscode/extensions/ms-python.vscode-pylance-2026.2.1/dist/server.bundle.js --cancellationReceive=file:3030fb1b3dfade6b14d745a5da2679f5b63ae9a4c9 --node-ipc --clientProcessId=4684',
   'connection_file': None,
   'is_current_process': False,
   'is_notebook_kernel': False},
  {'pid': 4684,
   'ppid': 4398,
   'rss_mb': 533.7,
   'command': '/snap/code/247/usr/share/code/code --type=utility --utility-sub-type=node.mojom.NodeService --lang=en-US --service-sandbox-type=none --no-sandbox --dns-result-order=ipv4first --experimental-network-inspection --inspect-por

In [2]:
import importlib
import json

import jax
from tqdm.auto import tqdm

from ant_byte_env.experiments import config_args_to_argv
from ant_byte_env.runs import write_json
from ant_byte_env.training.jax_mappo.timed_release import evaluation as timed_evaluation
from ant_byte_env.training.jax_mappo.timed_release import rendering as timed_rendering
from ant_byte_env.training.jax_mappo.timed_release import runner as timed_runner

workflows = importlib.reload(workflows)
timed_evaluation = importlib.reload(timed_evaluation)
timed_rendering = importlib.reload(timed_rendering)
timed_runner = importlib.reload(timed_runner)
print(f"JAX device: {jax.devices()[0]}")


JAX device: TFRT_CPU_0


In [3]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "timed_release_roles_8ants_shared_writes.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_NAME = experiment.name
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
EVAL_DIR = RUN_DIR / "evaluation"
for directory in (CHECKPOINT_DIR, MEDIA_DIR, EVAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Restore the source checkpoint first: {SOURCE_CHECKPOINT}")
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
BEST_CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

UPDATE_TIMESTEPS = int(experiment_args["num_envs"]) * int(experiment_args["num_steps"])
TOTAL_UPDATES = int(experiment_args["total_timesteps"]) // UPDATE_TIMESTEPS
CHUNK_UPDATES = int(experiment.metadata.get("chunk_updates", 100))
EVALUATION_EPISODES = int(experiment.metadata.get("evaluation_episodes", 4))
RENDER_ACTION_MODE = str(experiment.metadata.get("render_action_mode", "sampled_move_greedy_write"))
RENDER_MAX_FRAMES = int(experiment.metadata.get("render_max_frames", 480))
RENDER_TILE_SIZE = int(experiment.metadata.get("render_tile_size", workflows.NOTEBOOK_ROLLOUT_TILE_SIZE))

RUN_TRAINING = True
MAX_CHUNKS_TO_RUN = None
RESUME_FROM_BEST_CHECKPOINT = False
RUN_BEST_EVAL_DURING_TRAINING = False
ACTIVE_CHECKPOINT = (
    BEST_CHECKPOINT_PATH
    if RESUME_FROM_BEST_CHECKPOINT and BEST_CHECKPOINT_PATH.exists()
    else SOURCE_CHECKPOINT
)

summary = {
    "experiment": experiment.name,
    "source_checkpoint": str(SOURCE_CHECKPOINT),
    "run_dir": str(RUN_DIR),
    "total_updates": TOTAL_UPDATES,
    "chunk_updates": CHUNK_UPDATES,
    "release_interval": experiment_args["release_interval"],
    "initial_active_ants": experiment_args["initial_active_ants"],
    "max_steps": experiment_args["max_steps"],
    "num_envs": experiment_args["num_envs"],
    "num_steps": experiment_args["num_steps"],
    "critic_architecture": experiment_args["critic_architecture"],
    "actor_only_warm_start": experiment_args.get("actor_only_warm_start", False),
    "active_checkpoint": str(ACTIVE_CHECKPOINT),
    "training_rollout_temperature": experiment_args["training_rollout_temperature"],
    "eval_move_temperature": experiment_args["best_eval_move_temperature"],
    "run_best_eval_during_training": RUN_BEST_EVAL_DURING_TRAINING,
}
print(json.dumps(summary, indent=2))


{
  "experiment": "timed_release_roles_8ants_shared_writes",
  "source_checkpoint": "/home/juan/Documents/rl/cool-antz/runs/notebooks/exploration_to_forage_proximity_sources_full_layout_50x50_8ants_half_food_2src_shared_writes_from_64env_best/checkpoints/best_full_layout_proximity_8ants_half_food_shared_writes.pkl",
  "run_dir": "/home/juan/Documents/rl/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes",
  "total_updates": 500,
  "chunk_updates": 100,
  "release_interval": 150,
  "initial_active_ants": 1,
  "max_steps": 2000,
  "num_envs": 12,
  "num_steps": 96,
  "critic_architecture": "mlp",
  "actor_only_warm_start": true,
  "active_checkpoint": "/home/juan/Documents/rl/cool-antz/runs/notebooks/exploration_to_forage_proximity_sources_full_layout_50x50_8ants_half_food_2src_shared_writes_from_64env_best/checkpoints/best_full_layout_proximity_8ants_half_food_shared_writes.pkl",
  "training_rollout_temperature": 0.75,
  "eval_move_temperature": 0.52,
  "run_best_eval_duri

In [4]:
def progress_bar(label, total_updates):
    bar = tqdm(total=total_updates, desc=label)
    last = 0

    def callback(update, total, metrics):
        nonlocal last
        bar.total = total
        bar.update(max(0, int(update) - last))
        last = int(update)
        bar.set_postfix(
            ret=f"{metrics.get('episode_return', 0.0):.2f}",
            active=f"{metrics.get('mean_active_ants', 0.0):.2f}",
            delivered=f"{metrics.get('eval_mean_delivered_fraction', 0.0):.2f}",
        )

    return bar, callback

if RUN_TRAINING:
    previous_checkpoint = ACTIVE_CHECKPOINT
    chunk_count = (TOTAL_UPDATES + CHUNK_UPDATES - 1) // CHUNK_UPDATES
    if MAX_CHUNKS_TO_RUN is not None:
        chunk_count = min(chunk_count, int(MAX_CHUNKS_TO_RUN))
    chunk_metrics = []
    completed_updates = 0
    for chunk_index in range(chunk_count):
        updates = min(CHUNK_UPDATES, TOTAL_UPDATES - completed_updates)
        if updates <= 0:
            break
        label = f"chunk_{chunk_index + 1:03d}_updates_{completed_updates:05d}_{completed_updates + updates:05d}"
        chunk_dir = RUN_DIR / label
        chunk_checkpoint = CHECKPOINT_DIR / f"{label}.pkl"
        chunk_args = dict(experiment_args)
        chunk_args["total_timesteps"] = updates * UPDATE_TIMESTEPS
        chunk_args["load_model"] = str(previous_checkpoint)
        chunk_args["save_model"] = str(chunk_checkpoint)
        if RUN_BEST_EVAL_DURING_TRAINING:
            chunk_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)
        else:
            chunk_args["save_best_model"] = None
            chunk_args["best_model_selection"] = "train"
        chunk_args["run_dir"] = str(chunk_dir)
        argv = config_args_to_argv(chunk_args)
        bar, callback = progress_bar(label, updates)
        try:
            metrics = timed_runner.main(argv, progress_callback=callback)
        finally:
            bar.close()
        chunk_metrics.append({"label": label, "checkpoint": str(chunk_checkpoint), **metrics})
        previous_checkpoint = (
            BEST_CHECKPOINT_PATH
            if RUN_BEST_EVAL_DURING_TRAINING and BEST_CHECKPOINT_PATH.exists()
            else chunk_checkpoint
        )
        completed_updates += updates
    ACTIVE_CHECKPOINT = previous_checkpoint
    write_json(RUN_DIR / "training_chunks.json", {"chunks": chunk_metrics, "active_checkpoint": str(ACTIVE_CHECKPOINT)})
else:
    chunk_metrics = []

ACTIVE_CHECKPOINT


chunk_001_updates_00000_00100:   0%|          | 0/100 [00:00<?, ?it/s]

chunk_002_updates_00100_00200:   0%|          | 0/100 [00:00<?, ?it/s]

chunk_003_updates_00200_00300:   0%|          | 0/100 [00:00<?, ?it/s]

chunk_004_updates_00300_00400:   0%|          | 0/100 [00:00<?, ?it/s]

chunk_005_updates_00400_00500:   0%|          | 0/100 [00:00<?, ?it/s]

PosixPath('/home/juan/Documents/rl/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes/checkpoints/chunk_005_updates_00400_00500.pkl')

In [5]:
eval_metrics = timed_evaluation.evaluate_checkpoint(
    ACTIVE_CHECKPOINT,
    num_episodes=EVALUATION_EPISODES,
    action_mode=RENDER_ACTION_MODE,
    move_temperature=float(experiment_args.get("best_eval_move_temperature", 0.52)),
    write_temperature=float(experiment_args.get("best_eval_write_temperature", 1.0)),
)
eval_path = EVAL_DIR / f"timed_release_eval_{EVALUATION_EPISODES}ep.json"
write_json(eval_path, {"checkpoint": str(ACTIVE_CHECKPOINT), "metrics": eval_metrics})
eval_path, eval_metrics


(PosixPath('/home/juan/Documents/rl/cool-antz/runs/notebooks/timed_release_roles_8ants_shared_writes/evaluation/timed_release_eval_4ep.json'),
 {'eval_success_rate': 0.0,
  'eval_mean_delivered_food': 11.25,
  'eval_mean_delivered_fraction': 0.09,
  'eval_mean_episode_return': 11.25,
  'eval_mean_episode_length': 2000.0,
  'eval_mean_active_ant_steps': 11800.0,
  'eval_mean_delivered_food_per_1000_active_ant_steps': 0.9533898830413818,
  'eval_mean_pickups': 14.5,
  'eval_rank_0_mean_pickups': 2.5,
  'eval_rank_0_mean_deliveries': 2.0,
  'eval_rank_0_mean_writes': 1988.5,
  'eval_rank_0_mean_unique_cells': 175.25,
  'eval_rank_0_mean_first_pickup_step': 482.0,
  'eval_rank_0_mean_first_delivery_step': 132.75,
  'eval_rank_0_mean_release_to_pickup_latency': 482.0,
  'eval_rank_1_mean_pickups': 2.5,
  'eval_rank_1_mean_deliveries': 2.0,
  'eval_rank_1_mean_writes': 1842.0,
  'eval_rank_1_mean_unique_cells': 168.75,
  'eval_rank_1_mean_first_pickup_step': 496.75,
  'eval_rank_1_mean_first

In [ ]:
video_path = timed_rendering.render_timed_release_checkpoint(
    ACTIVE_CHECKPOINT,
    MEDIA_DIR / "timed_release_roles_rollout.mp4",
    max_frames=RENDER_MAX_FRAMES,
    tile_size=RENDER_TILE_SIZE,
    action_mode=RENDER_ACTION_MODE,
    move_temperature=float(experiment_args.get("best_eval_move_temperature", 0.52)),
    write_temperature=float(experiment_args.get("best_eval_write_temperature", 1.0)),
)
video_path
